# NumCompute quickstart

End-to-end demo of the `numcompute` package.

In [ ]:
%pip install -e ..

In [ ]:
import numpy as np
import numcompute

## Load, inspect, clean and prepare the data

This part of the demo uses three modules: `io` (loading), `utils`
(helpers) and `preprocessing` (cleaning). 

In [ ]:
from numcompute import io as nc_io, stats
from numcompute.preprocessing import (
    StandardScaler, SimpleImputer, OneHotEncoder)

path = "data/penguins.csv"
names = nc_io.read_header(path)
for i, name in enumerate(names):
    print(i, name)

### Load the file
`load_mixed_csv` reads every value as text first, then converts. It returns
two arrays: 
- `X_num` : (numbers, with NaN where a value was missing) 
- `X_cat` : (text categories).

In [ ]:
num_cols = [2, 3, 4, 5]    
cat_cols = [0, 1, 6] 

X_num, X_cat = nc_io.load_mixed_csv(path, num_cols, cat_cols, skip_header=1)

num_names = [names[i] for i in num_cols]
cat_names = [names[i] for i in cat_cols]

print("numeric    :", num_names, X_num.shape, X_num.dtype)
print("categorical:", cat_names, X_cat.shape, X_cat.dtype)

In [ ]:
print(f"{'':>3}" + "".join(f"{n[:15]:>17}" for n in num_names)
      + "".join(f"{n:>12}" for n in cat_names))
for i in range(5):
    print(f"{i:>3}" + "".join(f"{v:>17}" for v in X_num[i])
          + "".join(f"{v:>12}" for v in X_cat[i]))

### Missing data
Missing numbers must be filled before scaling.

In [ ]:
missing = np.isnan(X_num)
print("missing per column:", dict(zip(num_names, missing.sum(axis=0).tolist())))
rows_with_gaps = np.flatnonzero(missing.any(axis=1))
print("rows with a missing number:", rows_with_gaps.size, "of", X_num.shape[0])
print("row numbers:", rows_with_gaps.tolist())

### Summary of the numbers and the categories
For each measurement: mean, median, spread (standard deviation), smallest and
largest value, calculated with `numcompute.stats` while ignoring `nan`.

In [ ]:
for j, name in enumerate(cat_names):
    values, counts = np.unique(X_cat[:, j], return_counts=True)
    print(name, dict(zip(values.tolist(), counts.tolist())))

### Fill missing numbers.

`SimpleImputer` replaces each missing number
with the median of its column. The median is a good choice because
extreme values do not affect it much.

In [ ]:
first_gap = int(rows_with_gaps[0])
imputer = SimpleImputer("median").fit(X_num)
filled = imputer.transform(X_num)

print("missing numbers left:", int(np.isnan(filled).sum()))
print("medians used:", dict(zip(num_names, imputer.statistics_.round(1).tolist())))
print(f"row {first_gap} before:", X_num[first_gap])
print(f"row {first_gap} after :", filled[first_gap])

### Scale the numbers.

`StandardScaler` rescales each column so it has
mean 0 and standard deviation 1.

In [ ]:
scaler = StandardScaler().fit(filled)
Z = scaler.transform(filled)

print("mean per column:", Z.mean(axis=0).round(3))
print("std per column :", Z.std(axis=0).round(3))
print("row 0 raw   :", filled[0])
print("row 0 scaled:", Z[0].round(2))
print("undo scaling works:", np.allclose(scaler.inverse_transform(Z), filled))

### Encode the categories.
`OneHotEncoder` gives every category its own 0/1 column.

In [ ]:
enc = OneHotEncoder().fit(X_cat)
H = enc.transform(X_cat)

for name, cats in zip(cat_names, enc.categories_):
    print(name, "->", cats.tolist())
print("encoded shape:", H.shape)
print("row 0:", X_cat[0].tolist(), "->", H[0])
print("every row has one 1 per column:",
      bool(np.all(H.sum(axis=1) == len(cat_cols))))

In [ ]:
features = np.hstack([Z, H])
print("feature matrix:", features.shape)